### 📌 Rappel : à quoi servent les paramètres

| Paramètre | Valeur en dev | Sert à… | Où le voir |
|---|---|---|---|
| `CATALOG` | `dev` | indiquer **dans quel catalog** écrire. Le chemin de la landing est construit avec : `/Volumes/<CATALOG>/landing/files/…` | **Catalog** (menu de gauche) › `dev` |
| `SECRET_SCOPE` | `scope-data-dev` | indiquer **dans quel coffre** lire la clé API | pas visible dans l'interface : `dbutils.secrets.listScopes()` |
| `LEAGUE_IDS` | `61,39,78,140,135,2` | choisir les **championnats** : Ligue 1, Premier League, Bundesliga, Liga, Serie A, League des champions | documentation API-Football |
| `SEASONS` | `2022,2023,2024` | choisir les **saisons** (année de début). Le plan gratuit donne 2022 à 2024 | page **My Access** du site API-Football |

**Vocabulaire**
- **Catalog** : le plus grand contenant de Unity Catalog. Il regroupe les schémas (`landing`, `bronze`…), qui regroupent les tables et les volumes. Il y en a **un par environnement** :
  - `dev`, lié **uniquement** au workspace `dbx-dev` ;
  - `prod`, lié **uniquement** au workspace `dbx-prod`.
- **Secret scope** : un **coffre-fort** à secrets, propre à un workspace. C'est l'équivalent du Key Vault Azure.
- **`scope-data-dev`** : notre coffre dans `dbx-dev`. Il a été créé à la mission 3.4 – A. Les admins le gèrent (MANAGE) ; les développeurs et `sp-data-dev` le lisent (READ).
- **`api-football-key`** : le **nom du secret**, c'est-à-dire le tiroir du coffre qui contient la clé API. On le lit avec `dbutils.secrets.get(scope, key)`. Sa valeur s'affiche `[REDACTED]`.
- En prod (étape 6), on utilisera `CATALOG = prod` et `SECRET_SCOPE = scope-data-prod`. **Le code ne change pas.**

- print(dbutils.secrets.listScopes())              # les coffres que tu peux voir
- print(dbutils.secrets.list("scope-data-dev"))    # les noms des secrets du coffre, jamais leurs valeurs

# API-Football : matchs (fixtures) → landing

**Rôle** : récupérer les matchs de plusieurs championnats et saisons via l'API-Football et les déposer en **parquet** dans la landing.

**Destination** : `/Volumes/<CATALOG>/landing/files/api_football/fixtures/<aaaa>/<mm>/<jj>/<load_id>.<file_id>.parquet`
La date du chemin = **date du dépôt** (jour de l'appel à l'API), pas la date des matchs (colonne `fixture__date`).

| Paramètre (widget) | Exemple | Rôle |
|---|---|---|
| `CATALOG` | `dev` | catalog cible (dev / prod) |
| `SECRET_SCOPE` | `scope-data-dev` | coffre où est rangée la clé API |
| `LEAGUE_IDS` | `61,39,78,140,135,2` | championnats : Ligue 1, Premier League, Bundesliga, Liga, Serie A, League des champions |
| `SEASONS` | `2022,2023,2024` | saisons (année de début ; gratuit = 2022 à 2024) |

**Coût** : 1 requête par couple compétition × saison (ici 6 × 3 = **18**). L'appel `/status` ne compte pas.
**Rejouable** : oui ; chaque lancement **ajoute** un fichier (la bronze gérera les doublons).

## 1. Paramètres

In [0]:
dbutils.widgets.text("CATALOG", "dev", "Catalog cible")
dbutils.widgets.text("SECRET_SCOPE", "scope-data-dev", "Secret scope")
dbutils.widgets.text("LEAGUE_IDS", "61,39,78,140,135,2", "Ids championnats")
dbutils.widgets.text("SEASONS", "2022,2023,2024", "Saisons")

## 2. Installation de dlthub (version fixée)

In [0]:
%pip install "dlt[parquet]==1.31.0"

In [0]:
%restart_python

## 3. Import de dlthub + contrôle « fail fast » du conflit `dlt`

In [0]:
import dlt

if not hasattr(dlt, "__version__") or dlt.__file__.startswith("/databricks/"):
    raise ImportError("Conflit : le module 'dlt' chargé n'est pas dlthub (voir manuel, conflit dlt)")
print("✅ dlthub", dlt.__version__)

## 4. Configuration

- Les valeurs qui changent (catalog, coffre, championnats, saisons) viennent des **widgets**.
- La **clé API** vient du coffre : jamais écrite dans le code.
- L'adresse de l'API et la pause entre deux appels sont des constantes de la **source** (identiques en dev et en prod).

In [0]:
import os, time
os.environ["RUNTIME__DLTHUB_TELEMETRY"] = "false"   # pas d'envoi de statistiques d'usage à dlthub

from dlt.sources.helpers import requests   # requests avec relances automatiques en cas d'erreur réseau

def liste_entiers(texte: str) -> list[int]:
    """Transforme '61, 39,78' en [61, 39, 78]."""
    return [int(x) for x in texte.split(",") if x.strip()]

CATALOG      = dbutils.widgets.get("CATALOG").strip()
SECRET_SCOPE = dbutils.widgets.get("SECRET_SCOPE").strip()
LEAGUE_IDS   = liste_entiers(dbutils.widgets.get("LEAGUE_IDS"))
SEASONS      = liste_entiers(dbutils.widgets.get("SEASONS"))

BASE_URL        = "https://v3.football.api-sports.io"
PAUSE_SECONDES  = 7      # pause entre deux appels, pour respecter la limite par minute de l'API
HEADERS         = {"x-apisports-key": dbutils.secrets.get(scope=SECRET_SCOPE, key="api-football-key")}

LANDING = f"/Volumes/{CATALOG}/landing/files"
DATASET = "api_football"
NB_APPELS = len(LEAGUE_IDS) * len(SEASONS)

print(f"Catalog={CATALOG} | Scope={SECRET_SCOPE}")
print(f"Championnats={LEAGUE_IDS} | Saisons={SEASONS} | {NB_APPELS} appels prévus")
print(f"Landing : {LANDING}/{DATASET}/")

## 5. Contrôle du quota (`/status`, gratuit)

On vérifie **avant** de consommer des requêtes : compte actif et assez de requêtes restantes pour **tout le lot**.

In [0]:
statut = requests.get(f"{BASE_URL}/status", headers=HEADERS).json()["response"]

actif     = statut["subscription"]["active"]
utilisees = statut["requests"]["current"]
limite    = statut["requests"]["limit_day"]
restantes = limite - utilisees

print(f"Plan {statut['subscription']['plan']} | actif={actif} | {utilisees}/{limite} utilisées | {restantes} restantes")
if not actif or restantes < NB_APPELS:
    raise RuntimeError(f"Quota insuffisant : {NB_APPELS} requêtes nécessaires, {restantes} restantes. Chargement annulé.")

## 6. La ressource dlthub : les matchs

Une **ressource** dlthub = une fonction qui *produit* des données (`yield`) et le nom de la table où elles iront.
- Une boucle appelle l'API pour chaque couple **championnat × saison**, avec une pause entre deux appels.
- `write_disposition="append"` : chaque lancement **ajoute** (landing = historique des dépôts).
- On contrôle le champ `errors` de l'API : s'il n'est pas vide (ex. saison non autorisée), on **s'arrête** avec le message de l'API.
- Chaque match contient déjà `league.id` et `league.season` : on saura d'où vient chaque ligne (colonnes `league__id`, `league__season`).

In [0]:
@dlt.resource(name="fixtures", write_disposition="append")
def fixtures(league_ids: list[int], seasons: list[int]):
    for i, (league_id, season) in enumerate((l, s) for l in league_ids for s in seasons):
        if i > 0:
            time.sleep(PAUSE_SECONDES)

        reponse = requests.get(
            f"{BASE_URL}/fixtures",
            headers=HEADERS,
            params={"league": league_id, "season": season},
        ).json()

        if reponse.get("errors"):
            raise RuntimeError(f"Erreur API-Football (ligue {league_id}, saison {season}) : {reponse['errors']}")

        print(f"📥 ligue {league_id} | saison {season} : {reponse['results']} matchs")
        yield reponse["response"]

## 7. Le pipeline et le chargement

- `bucket_url` : la racine de la landing ; `dataset_name` : le dossier de la source (`api_football`).
- `layout` : notre convention `<table>/<aaaa>/<mm>/<jj>/` (date du dépôt). dlthub impose que `{table_name}` soit **avant** la date.
- Le JSON imbriqué est **aplati** par dlthub : `fixture.referee` → colonne `fixture__referee`, `teams.home.name` → `teams__home__name`, etc.

In [0]:
pipeline = dlt.pipeline(
    pipeline_name="api_football_fixtures",
    destination=dlt.destinations.filesystem(
        bucket_url=f"file://{LANDING}",
        layout="{table_name}/{YYYY}/{MM}/{DD}/{load_id}.{file_id}.{ext}",
    ),
    dataset_name=DATASET,
)

info = pipeline.run(fixtures(LEAGUE_IDS, SEASONS), loader_file_format="parquet")
print(info)
info.raise_on_failed_jobs()      # s'arrête en erreur si un fichier n'a pas pu être écrit

## 8. Vérification : fichiers déposés aujourd'hui

In [0]:
from datetime import datetime, timezone

aujourdhui = datetime.now(timezone.utc)
dossier_jour = f"{LANDING}/{DATASET}/fixtures/{aujourdhui:%Y/%m/%d}"

for f in sorted(os.listdir(dossier_jour)):
    print("📄", f)

## 9. Vérification : contenu

- Nombre de matchs par **championnat** et par **saison** (contrôle que chaque id correspond bien au championnat attendu).
- Aperçu de quelques colonnes clés (date, arbitre, équipes, score).

In [0]:
from pyspark.sql import functions as F

df = spark.read.parquet(dossier_jour)
print("Colonnes :", len(df.columns), "| Lignes (tous dépôts du jour) :", df.count())

display(df.groupBy("league__id", "league__name", "league__country", "league__season")
          .agg(F.count("*").alias("nb_matchs"))
          .orderBy("league__id", "league__season"))

In [0]:
display(df.select(
    "league__name", "league__season", "fixture__id", "fixture__date", "fixture__referee",
    "teams__home__name", "teams__away__name", "goals__home", "goals__away", "_dlt_load_id",
).orderBy("league__name", "fixture__date").limit(20))

## 10. Quota après chargement

In [0]:
apres = requests.get(f"{BASE_URL}/status", headers=HEADERS).json()["response"]["requests"]
print(f"📊 {apres['current']}/{apres['limit_day']} requêtes utilisées aujourd'hui")

In [0]:
from pyspark.sql import functions as F

df = spark.read.parquet(dossier_jour)

def equipes(cote):   # cote = "home" ou "away"
    return df.select(
        F.col("league__id").alias("league_id"), F.col("league__name").alias("competition"),
        F.col("league__season").alias("saison"),
        F.col(f"teams__{cote}__id").alias("team_id"), F.col(f"teams__{cote}__name").alias("equipe"),
    )

toutes = equipes("home").unionByName(equipes("away")).distinct()

championnats = toutes.filter(F.col("league_id").isin(61, 39, 78, 140, 135)).withColumnRenamed("competition", "championnat")
coupes       = toutes.filter(F.col("league_id").isin(2, 3, 848)).select("team_id", "saison", F.col("competition").alias("coupe"))

croisement = championnats.join(coupes, ["team_id", "saison"])

display(croisement.groupBy("saison", "championnat", "coupe")
                  .agg(F.countDistinct("team_id").alias("nb_equipes"))
                  .orderBy("saison", "championnat", "coupe"))

In [0]:
from pyspark.sql import functions as F
display(spark.read.parquet(dossier_jour)
        .groupBy("league__id", "league__name", "league__season")
        .agg(F.count("*").alias("nb_matchs"))
        .orderBy("league__id", "league__season"))